In [3]:

import pandas as pd

df = pd.read_csv("googleplaystore.csv")
print(df.head())
print("Shape:", df.shape)

reviews_df = pd.read_csv("googleplaystore_user_reviews.csv")
print(reviews_df.head())
print("Reviews Shape:", reviews_df.shape)


df.columns = df.columns.str.strip()
reviews_df.columns = reviews_df.columns.str.strip()


                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres      Last Updated         Current Ver  \
0               Art & Design   January 7, 2018               1.0.0   
1  Art & Design;Pretend 

In [4]:

if "Sentiment_Polarity" in reviews_df.columns:
    sentiment_col = "Sentiment_Polarity"
else:
    raise ValueError("Sentiment_Polarity في user_reviews")


sentiment_mean = reviews_df.groupby("App")[sentiment_col].mean().reset_index()
sentiment_mean.columns = ["App", "Avg_Sentiment"]


df = df.merge(sentiment_mean, on="App", how="left")

print(df.head().to_string())


                                                  App        Category  Rating Reviews  Size     Installs  Type Price Content Rating                     Genres      Last Updated         Current Ver   Android Ver  Avg_Sentiment
0      Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1     159   19M      10,000+  Free     0       Everyone               Art & Design   January 7, 2018               1.0.0  4.0.3 and up            NaN
1                                 Coloring book moana  ART_AND_DESIGN     3.9     967   14M     500,000+  Free     0       Everyone  Art & Design;Pretend Play  January 15, 2018               2.0.0  4.0.3 and up       0.152652
2  U Launcher Lite – FREE Live Cool Themes, Hide Apps  ART_AND_DESIGN     4.7   87510  8.7M   5,000,000+  Free     0       Everyone               Art & Design    August 1, 2018               1.2.4  4.0.3 and up            NaN
3                               Sketch - Draw & Paint  ART_AND_DESIGN     4.5  215644   25M  50,

In [5]:

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df.dropna(subset=["Rating"])


df["Installs"] = df["Installs"].astype(str).str.replace("+", "", regex=False)
df["Installs"] = df["Installs"].str.replace(",", "", regex=False)
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")


df["Price"] = df["Price"].astype(str).str.replace("$", "", regex=False)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")


df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")


def convert_size(x):
    if isinstance(x, str):
        x = x.strip()
        if x.endswith("M"):
            try:
                return float(x.replace("M", "")) * 1_000_000
            except:
                return None
        elif x.endswith("k") or x.endswith("K"):
            try:
                return float(x.replace("k", "").replace("K", "")) * 1_000
            except:
                return None
        elif x == "Varies with device":
            return None
        else:

            try:
                return float(x)
            except:
                return None
    else:
        return x

df["Size"] = df["Size"].apply(convert_size)


numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

preview_numeric = df[numeric_cols].head()
print(preview_numeric.to_string(index=False))



 Rating  Reviews       Size   Installs  Price  Avg_Sentiment
    4.1    159.0 19000000.0    10000.0    0.0       0.181818
    3.9    967.0 14000000.0   500000.0    0.0       0.152652
    4.7  87510.0  8700000.0  5000000.0    0.0       0.181818
    4.5 215644.0 25000000.0 50000000.0    0.0       0.181818
    4.3    967.0  2800000.0   100000.0    0.0       0.181818


In [6]:

df["Is_Free"] = (df["Price"] == 0).astype(int)

df["Reviews_per_Install"] = df["Reviews"] / df["Installs"]
df["Reviews_per_Install"] = df["Reviews_per_Install"].replace([float("inf")], 0).fillna(0)


import pandas as pd
from datetime import datetime

df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
max_date = df["Last Updated"].max()
df["Days_Since_Update"] = (max_date - df["Last Updated"]).dt.days
df["Days_Since_Update"] = df["Days_Since_Update"].fillna(df["Days_Since_Update"].median())

print("\n===== Feature Engineering Preview =====")
print(df[["Price","Is_Free","Reviews_per_Install","Days_Since_Update"]].head().to_string(index=False))




===== Feature Engineering Preview =====
 Price  Is_Free  Reviews_per_Install  Days_Since_Update
   0.0        1             0.015900              213.0
   0.0        1             0.001934              205.0
   0.0        1             0.017502                7.0
   0.0        1             0.004313               61.0
   0.0        1             0.009670               49.0


In [7]:

df["HighRating"] = (df["Rating"] >= 4.0).astype(int)

drop_cols = [
    "App",
    "Rating",
    "HighRating",
    "Last Updated",
    "Current Ver",
    "Android Ver"
]

for col in drop_cols:
    if col in df.columns:
        pass
    else:
        print(f"Warning: {col} not found in df.columns")


X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df["HighRating"]

print("X shape before encoding:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True))


X shape before encoding: (9367, 12)
y distribution:
 HighRating
1    0.786698
0    0.213302
Name: proportion, dtype: float64


In [8]:

X = pd.get_dummies(X, drop_first=True)

print("X shape after encoding:", X.shape)


print("Any object columns left?", len(X.select_dtypes(include=["object"]).columns))


X shape after encoding: (9367, 163)
Any object columns left? 0


In [9]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


Train shape: (7493, 163)
Test shape : (1874, 163)


In [10]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

rf_selector = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

selector = SelectFromModel(rf_selector)

selector.fit(X_train, y_train)

X_train_sfm = selector.transform(X_train)
X_test_sfm = selector.transform(X_test)

print("Before SelectFromModel:", X_train.shape)
print("After SelectFromModel :", X_train_sfm.shape)


Before SelectFromModel: (7493, 163)
After SelectFromModel : (7493, 11)


In [11]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
    else:
        y_score = None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    if y_score is not None:
        roc = roc_auc_score(y_test, y_score)
    else:
        roc = None

    cm = confusion_matrix(y_test, y_pred)

    print(f"\n===== {name} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1       : {f1:.4f}")
    if roc is not None:
        print(f"ROC AUC  : {roc:.4f}")
    else:
        print("ROC AUC  : N/A (no score)")
    print("Confusion Matrix:\n", cm)

    return {
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": roc
    }


In [12]:

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=2000, n_jobs=-1)
result_lr = evaluate_model(
    lr_model,
    X_train_sfm, X_test_sfm,
    y_train, y_test,
    "LR"
)



===== LR =====
Accuracy : 0.7860
Precision: 0.7867
Recall   : 0.9986
F1       : 0.8801
ROC AUC  : 0.6485
Confusion Matrix:
 [[   1  399]
 [   2 1472]]


In [13]:

from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=None
)
result_dt = evaluate_model(
    dt_model,
    X_train_sfm, X_test_sfm,
    y_train, y_test,
    "DT"
)



===== DT =====
Accuracy : 0.7252
Precision: 0.8314
Recall   : 0.8161
F1       : 0.8237
ROC AUC  : 0.6031
Confusion Matrix:
 [[ 156  244]
 [ 271 1203]]


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
result_rf = evaluate_model(
    rf_model,
    X_train_sfm, X_test_sfm,
    y_train, y_test,
    "RF"
)



===== RF =====
Accuracy : 0.7855
Precision: 0.8229
Recall   : 0.9267
F1       : 0.8717
ROC AUC  : 0.7798
Confusion Matrix:
 [[ 106  294]
 [ 108 1366]]


In [20]:
results_muaiyed = pd.DataFrame([result_lr, result_dt, result_rf])
results_muaiyed

print("\n=== muaiyed Models Results ===")
print(results_muaiyed.to_string(index=False))



=== muaiyed Models Results ===
Model  Accuracy  Precision   Recall       F1  ROC_AUC
   LR  0.786019   0.786745 0.998643 0.880120 0.648519
   DT  0.725187   0.831375 0.816147 0.823691 0.603073
   RF  0.785486   0.822892 0.926730 0.871729 0.779774
